# Modules 1–2: First Trace + Build the Survey-Feedback Agent

> Part 2 of the workshop, ~25 min (first trace ~10 min, build ~15 min). Everything runs against **your own LangSmith instance**.

**The use case.** Customer survey exports pile up faster than survey owners can read them. We'll build an agent that reads an export, measures NPS, clusters comments into themes backed by **verbatim quotes**, drafts a summary, and posts it to Slack or Teams, but only after a human approves.

| Module | What you'll do |
|---|---|
| **1. First trace** | Register the agent, send traces to its **development** environment, mask sensitive data before it leaves the app, and see the agent in the **Agents** view. |
| **2. Build** | Use Deep Agents to add a filesystem for survey exports, tools, a **theme-clustering subagent**, and **human-in-the-loop approval** before a summary is sent. Inspect the nested traces as you go. Then package it as a **Managed Deep Agent** and see what the managed layer adds. |

The finished agent is a **Managed Deep Agents** project in `survey_agent/`. Section 2.5 packages it and covers what Managed Deep Agents adds on top of the Deep Agent you build here; Module 3 deploys it with `mda deploy`.

## Setup

In [ ]:
import os, sys, time
from pathlib import Path

project_root = Path().resolve().parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from dotenv import load_dotenv
load_dotenv(project_root / ".env", override=True)

# Agent addressing and LANGSMITH_PROJECT can't be combined: with both set, the SDK
# can't pick a destination and drops the trace.
if os.environ.pop("LANGSMITH_PROJECT", None):
    print("Note: ignoring LANGSMITH_PROJECT in this kernel (conflicts with agent addressing).")

import warnings
# Keep outputs readable: agent addressing is in beta (one-time notice), and
# list_runs() is deprecated but supported until 2027.
warnings.filterwarnings("ignore", message="Addressing runs with `langsmith.address` is in beta")
warnings.filterwarnings("ignore", message=r"list_runs\(\) is deprecated")

import langsmith as ls
from langsmith import Client, uuid7

from utils.models import model
from utils.agent_registry import ensure_agent, environment_project_name

AGENT_ID = "survey-feedback-agent"   # the agent's identifier in LangSmith (lowercase, hyphens)
ENVIRONMENT = "development"
dev_address = ls.agent_address(AGENT_ID, ENVIRONMENT)

client = Client()
# Models go through the LangSmith LLM Gateway. `langsmith:<provider>/<model>` models
# read LANGSMITH_GATEWAY_API_KEY (then LANGSMITH_API_KEY); reuse the workshop's gateway key.
if os.environ.get("LANGSMITH_API_KEY_GATEWAY"):
    os.environ.setdefault("LANGSMITH_GATEWAY_API_KEY", os.environ["LANGSMITH_API_KEY_GATEWAY"])
for key in ["LANGSMITH_API_KEY", "LANGSMITH_API_KEY_GATEWAY"]:
    print(f"{key:26s} set: {bool(os.environ.get(key))}")

---
# Module 1. First Trace

In an agent-based LangSmith workspace, every trace belongs to an **agent** and one of its four **environments**: Local, Development, Staging, Production. Traces, evaluators, alerts and prebuilt dashboards are all scoped to one environment of one agent, so dev traffic never pollutes production alerts.

> Agent-based workspaces are a LangSmith Cloud feature. If the control at the top left of LangSmith shows the LangSmith logo instead of your workspace name, your workspace is project-based. In that case, ask your admin to migrate it before the session.

### 1.1 Register the agent

Create the agent record once. Its **identifier** (`survey-feedback-agent`) can't change later and is what code uses to address traces. Its display name can. Creating an agent needs a key with `projects:create` permission.

In [ ]:
agent_record = ensure_agent(client, AGENT_ID, name="Survey Feedback Agent")
print("Agent:", agent_record.id, "|", agent_record.name)
for env in agent_record.environments:
    print(f"  {env.environment:12s} trace storage: {env.tracer_session_name}")

# Some APIs (run queries, run rules) still take the environment's project name.
dev_project = environment_project_name(client, AGENT_ID, ENVIRONMENT)

### 1.2 Send a trace to the development environment

`ls.agent_address(agent_id, environment)` names the destination. Pass it to `tracing_context` and everything traced inside the block lands there: `@traceable` functions, LangChain model calls, and agents.

In [ ]:
from langsmith import traceable
from survey_agent.tools.survey import load_survey_export


@traceable(name="summarize_one_response")
def summarize_one_response(comment: str) -> str:
    reply = model.invoke([
        ("system", "Summarize this customer survey comment in one sentence and label its sentiment."),
        ("user", comment),
    ])
    return reply.text


sample = load_survey_export()[1]["comment"]
with ls.tracing_context(address=dev_address, client=client):
    print(summarize_one_response(sample))
client.flush()

**Find it in LangSmith:** select **Survey Feedback Agent** in the agent switcher (top bar), open **Tracing**, and set the environment control to **Development**. The trace shows the `summarize_one_response` root with the model call nested under it.

Open the **Agents** view as well. The agent is listed with its environments, and this list is where the Fleet and ADK agents appear later. It's the starting point for an agent registry.

### 1.3 Mask sensitive inputs and outputs before they leave the app

Survey comments sometimes contain emails and phone numbers. `Client(hide_inputs=..., hide_outputs=...)` runs your function on every run's inputs and outputs **in the app process, before anything is sent**, so the raw values never reach LangSmith.

In [ ]:
import re

EMAIL = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
PHONE = re.compile(r"\(?\b\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}\b")


def redact(value):
    """Recursively replace emails and phone numbers in run inputs/outputs."""
    if isinstance(value, str):
        return PHONE.sub("[PHONE]", EMAIL.sub("[EMAIL]", value))
    if isinstance(value, dict):
        return {k: redact(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [redact(v) for v in value]
    return value


masked_client = Client(hide_inputs=redact, hide_outputs=redact)

pii_comment = next(r["comment"] for r in load_survey_export() if r["response_id"] == "R-1012")
print("Raw comment in the app:\n ", pii_comment, "\n")

with ls.tracing_context(address=dev_address, client=masked_client):
    summarize_one_response(pii_comment)
masked_client.flush()

In [ ]:
# Read the run back from LangSmith: the stored inputs are already redacted.
time.sleep(5)
latest = next(iter(client.list_runs(
    project_name=dev_project, is_root=True, filter='eq(name, "summarize_one_response")', limit=1,
)))
print("Stored in LangSmith:\n ", latest.inputs)

> For blanket masking with no code, set `LANGSMITH_HIDE_INPUTS=true` / `LANGSMITH_HIDE_OUTPUTS=true`. Workspace-level redaction of model traffic is also available through the LangSmith Gateway. From here on, every run in this notebook goes through `masked_client`.

---
# Module 2. Build the Survey-Feedback Agent

Deep Agents = `create_agent()` + a prebuilt harness: a virtual filesystem, planning, subagents, context management, and human-in-the-loop. We'll build the agent in four steps, tracing each run to the development environment:

1. A filesystem holding the survey exports
2. Tools for exact numbers and verbatim quotes
3. A theme-clustering subagent
4. Human approval before anything is posted

In [ ]:
from deepagents import create_deep_agent
from deepagents.backends import CompositeBackend, FilesystemBackend, StateBackend
from langgraph.checkpoint.memory import MemorySaver
from langgraph.types import Command

from survey_agent.tools.survey import DATA_DIR, LANGCHAIN_TOOLS, get_survey_stats

SYSTEM = "You are a survey-feedback analyst for an electronics retailer."


def run(agent, prompt, config=None):
    """Invoke an agent with tracing addressed to the dev environment (masked)."""
    config = config or {"configurable": {"thread_id": str(uuid7())}}
    with ls.tracing_context(address=dev_address, client=masked_client):
        result = agent.invoke({"messages": [{"role": "user", "content": prompt}]}, config=config)
    masked_client.flush()
    return result, config

### 2.1 A filesystem for survey exports

The default **StateBackend** keeps files in the thread's state. A **CompositeBackend** mounts the real export directory (`survey_agent/data/`) at `/surveys/`, so the agent can `ls`, `read_file` and `grep` exports with its built-in tools.

> Backends are yours to choose with `create_deep_agent`. Managed Deep Agents owns the backend instead (section 2.5), so the packaged agent reads exports through a `read_survey_export` tool.

In [ ]:
def survey_backend():
    return CompositeBackend(
        default=StateBackend(),
        routes={"/surveys/": FilesystemBackend(root_dir=DATA_DIR, virtual_mode=True)},
    )


agent_v1 = create_deep_agent(
    model=model, system_prompt=SYSTEM, backend=survey_backend(), checkpointer=MemorySaver(),
)
result, _ = run(agent_v1, (
    "Look in /surveys/. How many responses does the export have, what columns does it "
    "contain, and what do customers complain about most?"
))
print(result["messages"][-1].text)

### 2.2 Tools: exact numbers, verbatim quotes

Reading a CSV and estimating NPS in its head is where an LLM drifts. Tools make the numbers deterministic and give the agent a way to pull exact quotes. These tools live in `survey_agent/tools/survey.py` and are shared with the ADK agent in Module 6, so both frameworks are scored on equal footing.

In [ ]:
tools = {t.name: t for t in LANGCHAIN_TOOLS}
for t in LANGCHAIN_TOOLS:
    print(f"- {t.name}: {t.description.splitlines()[0]}")

agent_v2 = create_deep_agent(
    model=model,
    system_prompt=SYSTEM + " Use get_survey_stats for every number; never estimate.",
    tools=[tools["get_survey_stats"], tools["search_responses"]],
    backend=survey_backend(),
    checkpointer=MemorySaver(),
)
result, _ = run(agent_v2, (
    "What is NPS overall and by channel? Back up the worst channel with two verbatim quotes."
))
print(result["messages"][-1].text)
print("\nGround truth from the tool:", get_survey_stats("channel")["groups"])

### 2.3 A theme-clustering subagent

Clustering takes many searches. A **subagent** does that work in its own context window and hands back only the result (themes, response IDs and quotes), which keeps the main agent's context clean. The main agent calls it with the `task` tool.

In [ ]:
from agents.local_agent import DEFINITION

THEME_SUBAGENT = DEFINITION["subagents"][0]  # the subagent declared in survey_agent/

print(THEME_SUBAGENT["system_prompt"])

agent_v3 = create_deep_agent(
    model=model,
    system_prompt=SYSTEM + " Use get_survey_stats for every number; never estimate.",
    tools=[tools["get_survey_stats"]],
    subagents=[THEME_SUBAGENT],
    backend=survey_backend(),
    checkpointer=MemorySaver(),
)
result, _ = run(agent_v3, (
    "Delegate theme clustering to the theme-clustering-agent, then give me the top 4 themes "
    "ranked by impact on NPS, each with two verbatim quotes cited by response ID."
))
print(result["messages"][-1].text)

**Inspect the nested trace.** In LangSmith, open the latest trace in Development. The `task` tool call contains the subagent's own model and tool calls (`search_responses`, `theme_counts`), nested one level down. The same tree, from code:

In [ ]:
time.sleep(5)
root = next(iter(client.list_runs(project_name=dev_project, is_root=True, limit=1)))
runs = list(client.list_runs(project_name=dev_project, trace_id=root.trace_id))
parent_of = {r.id: r.parent_run_id for r in runs}
names = {r.id: r.name for r in runs}


def inside_subagent(r):
    p = r.parent_run_id
    while p:
        if names.get(p) == "task":
            return True
        p = parent_of.get(p)
    return False


print(f"Trace: {root.name}")
for r in sorted(runs, key=lambda r: r.start_time):
    if r.run_type in ("llm", "tool"):
        indent = "      " if inside_subagent(r) else "  "
        print(f"{indent}[{r.run_type}] {r.name}")

### 2.4 Human-in-the-loop: approve before a summary is sent

`interrupt_on={"post_themes_to_channel": ...}` pauses the agent **before** the tool runs and saves the thread with a checkpointer. A reviewer can then **approve**, **edit** the message, or **reject** it with feedback, and the agent continues from where it stopped.

This is the packaged agent from the Managed Deep Agents project in `survey_agent/`, compiled in this kernel by `agents/local_agent.py` (section 2.5 explains how). It adds an `instructions.md` workflow and a `survey-summary` skill to what we built above. Locally we resume from code. Once deployed (Module 3), the same paused thread is what reviewers see in **Agent Inbox**.

In [ ]:
from agents.local_agent import build_agent

survey_agent = build_agent(interrupts=True, checkpointer=MemorySaver())
result, config = run(survey_agent, (
    "Analyze the latest survey export, draft the summary for the survey owners, "
    "and post it to #survey-insights."
))

pending = result.get("__interrupt__")
if pending:
    for action in pending[0].value["action_requests"]:
        print(f"Paused before {action['name']} -> {action['args']['channel']}\n")
        print(action["args"]["message"])
else:
    print("No approval requested. Final reply:\n", result["messages"][-1].text)

In [ ]:
# Decide. Approve as-is, or reject with feedback and the agent will revise.
decision = {"type": "approve"}
# decision = {"type": "reject", "message": "Lead with the delivery theme and keep it under 200 words."}

with ls.tracing_context(address=dev_address, client=masked_client):
    result = survey_agent.invoke(Command(resume={"decisions": [decision]}), config=config)
masked_client.flush()

if result.get("__interrupt__"):
    print("The agent revised the draft and is asking again. Re-run the previous cell's printout:")
    print(result["__interrupt__"][0].value["action_requests"][0]["args"]["message"])
else:
    print(result["messages"][-1].text)

In [ ]:
# Without SURVEY_WEBHOOK_URL in .env, approved posts are recorded locally instead of sent.
from survey_agent.tools.survey import OUTBOX
print("Outbox:", OUTBOX)
print(OUTBOX.read_text().splitlines()[-1] if OUTBOX.exists() else "(nothing posted)")

### 2.5 Package it as a Managed Deep Agent

Every agent so far was compiled in this kernel with `create_deep_agent`, which means **we own everything around the agent**: the `MemorySaver` checkpointer that makes approvals resumable, the backend that mounts `/surveys/`, and, once it ships, a server, its scaling, and its secrets.

**Managed Deep Agents (MDA)** runs the *same* Deep Agents harness (agent loop, filesystem tools, subagents, skills, `interrupt_on`), and LangSmith supplies the layer around it. You describe the agent as a project folder, and `mda deploy` turns it into a hosted agent. Here is the survey agent as an MDA project:

In [ ]:
survey_project = project_root / "survey_agent"
for p in sorted(survey_project.rglob("*")):
    rel = p.relative_to(survey_project)
    if p.is_file() and not any(part.startswith(".") or part == "__pycache__" for part in rel.parts) \
            and rel.name not in {"uv.lock", "README.md"}:
        print(" ", rel)

print("\n--- agent.py ---")
print((survey_project / "agent.py").read_text())

**What changed from `create_deep_agent` to `define_deep_agent`:**

| In this notebook (`create_deep_agent`) | In `survey_agent/` (Managed Deep Agents) |
|---|---|
| `tools=`, `subagents=`, `interrupt_on=` | **Unchanged**: same parameters on `define_deep_agent` |
| `system_prompt="..."` | `instructions.md`, synced to **Context Hub** on deploy |
| `skills=[...]` plus a backend route to the files | `skills/survey-summary/SKILL.md`, synced to **Context Hub** |
| `checkpointer=MemorySaver()` (yours to run in production) | Managed: durable checkpoints, so approvals survive restarts with no setup |
| `backend=CompositeBackend(...)` mounting `/surveys/` | Managed: thread filesystem, or a managed sandbox (`sandbox/`). No custom backends, so exports ship with the project and the agent reads them with `read_survey_export` |
| A server you build and run | `mda deploy` → Agent Server, bound to the agent's environment |
| — | `identity.py`: who may call the deployment (LangSmith API key by default) |

`define_deep_agent` returns a **definition**, not a compiled agent. The managed runtime compiles it with `create_deep_agent` at run time. `agents/local_agent.py` does the same compile in this kernel, which is how section 2.4 ran the packaged agent and how Module 4 evaluates it: **one definition, run locally or deployed**.

In [ ]:
from agents.local_agent import DEFINITION, INSTRUCTIONS_PATH

print("Definition parameters:", sorted(DEFINITION))
print("Tools:       ", [t.name for t in DEFINITION["tools"]])
print("Subagents:   ", [s["name"] for s in DEFINITION["subagents"]])
print("Approval on: ", list(DEFINITION["interrupt_on"]))
print("\n--- instructions.md (first lines) ---")
print("\n".join(INSTRUCTIONS_PATH.read_text().splitlines()[:6]))

`mda build` compiles the project into a deployable app without deploying it. It's a quick check that the definition, tools, and declarations load:

In [ ]:
# survey_agent/ is its own uv project; unset the notebook's VIRTUAL_ENV so uv uses that project's env.
!cd "{project_root}/survey_agent" && env -u VIRTUAL_ENV uv sync -q && env -u VIRTUAL_ENV uv run mda build

The build lands in `survey_agent/.mda/build/`: your code and data, a generated `langgraph.json` with one graph named `survey-feedback-agent` (the definition's `name`, which clients pass as `assistant_id`), and a runtime wrapper that injects the backend, checkpointer, identity, and sandbox. On deploy, `instructions.md` and `skills/` go to Context Hub rather than into the image.

### 2.6 What Managed Deep Agents adds on top of the Deep Agent we built

| Capability | With `create_deep_agent`, you build and run it | With Managed Deep Agents | For the survey agent |
|---|---|---|---|
| **Managed runtime** | Server, checkpointer, store, scaling | Agent Server with threads, runs, streaming, durable checkpoints | Approvals pause and resume for days with nothing to operate |
| **One-command deploy** | Packaging, images, secrets, environments | `mda deploy --name <agent> --agent-environment development\|staging\|production`; `.env` values become deployment secrets | Dev and staging deployments under one agent (Module 3) |
| **Context Hub** | Prompt and skills baked into code | `instructions.md` and `skills/` live in Context Hub; edit them without a rebuild | Survey owners tweak the summary format without a code deploy |
| **Channels** | Your own Slack app or webhook code | `channels/slack.py`: people @mention the agent in Slack, replies land in-thread | Ask the agent questions about the survey in `#survey-insights` |
| **Schedules** | A cron service you run | `schedules/<name>.py` with `define_schedule(...)` | A Monday-morning survey digest, still gated by approval |
| **LLM Gateway models** | Provider keys in every environment | `model="langsmith:openai/gpt-5.6-luna"`: one LangSmith key, Gateway spend, rate, and data policies | The survey agent ships with no provider key |
| **Connections** | Credential storage, OAuth callbacks, token refresh | `mda connections create ...` with agent-owned or per-user OAuth credentials | The Slack/Teams credential lives in the workspace, not in `.env` |
| **Identity** | Auth middleware | `identity.py`: LangSmith API key (default), Supabase, any JWT provider, or your backend | Lock the deployment to your workspace; per-user private threads for an end-user app |
| **Durable memory** | Wire up a store and memory paths | `memory.py` with `define_memory(agent=MemoryLayer())`, mounted at `/memories/agent/` | Remember last week's themes and report trends |
| **Sandbox** | Provision and clean up VMs | `sandbox/`: a per-thread filesystem and shell, created on demand and deleted when idle | Run pandas over large exports the tools don't cover |
| **MCP endpoint** | Write an MCP server | Every deployment serves `<API URL>/mcp` | Call the survey agent from Claude Code or another agent |
| **Local Studio** | `langgraph dev` plus config | `uv run mda dev`, with traces going to the agent's **Local** environment | Step through approvals before deploying |

Each capability is one file. The workshop project keeps only what the agenda needs, plus the `identity.py` and `sandbox/` declarations that `mda init` scaffolds. To add the others:

```python
# schedules/weekly_digest.py: a Monday 8am digest. The post still pauses for approval.
from managed_deepagents import define_schedule
schedule = define_schedule(
    cron="0 8 * * 1",
    timezone="America/Chicago",
    prompt="Summarize the latest survey export and post it to #survey-insights.",
)

# channels/slack.py: created by `uv run mda channels init slack`
from managed_deepagents import channels
channel = channels.slack(name="Survey Pulse", description="Answers questions about customer survey feedback")

# memory.py: durable memory shared by every caller of the deployment
from managed_deepagents import MemoryLayer, define_memory
memory = define_memory(agent=MemoryLayer())
```

**When to stay on `create_deep_agent`:** Managed Deep Agents hosts the agent only. If you need a custom backend, your own store or checkpointer, a Python `state_schema`, or custom HTTP routes around the agent, deploy the `create_deep_agent` graph as a custom LangSmith Deployment instead. It's the same harness either way.

### 2.7 (Optional) Try it in Studio with `mda dev`

In a terminal, from the workshop root:

```bash
cd survey_agent
uv run mda dev
```

`mda dev` compiles the project, runs it on a local Agent Server, and opens Studio. Ask it to post the summary, and Studio shows the paused `post_themes_to_channel` call to approve, edit, or reject. Local runs trace to the agent's **Local** environment, which works because section 1.1 already created the agent record. Stop it with `Ctrl+C`.

> `mda` reads keys from `survey_agent/.env` first, then your shell. Module 3 fills that file in before deploying.

## Recap

- **First trace**: the agent is registered; traces go to `survey-feedback-agent` / Development via `ls.agent_address`; emails and phone numbers are masked in-process before anything is sent.
- **Build**: a filesystem for exports, deterministic tools, a theme-clustering subagent, and an approval gate before posting, all visible as nested traces.
- **Managed Deep Agents**: the same agent as a project folder (`define_deep_agent` + `instructions.md` + `skills/` + `tools/`). The harness is unchanged; LangSmith runs the runtime, persistence, deploys, Context Hub, and optional channels, schedules, memory, identity, and sandbox.

**Next, Module 3:** `mda deploy` to the development environment, call the agent through the API, approve the post in Agent Inbox, and promote it to staging.